# Sección 2.1 — Limpieza de las variables predictoras operativas

Parte del `dataset_trabajo.csv` (1.172 días válidos, generado en `correccion_fechas.ipynb`), donde los días inválidos ya fueron filtrados mirando el target `Frio (Kw)`. Esta sección limpia las **demás columnas predictoras**, que en el cierre de las 23:59 todavía tienen errores de medidor y nulos.

**Regla general para evitar fuga de información:** todo lo que se aprende de los datos (qué variable representa a cada grupo colineal, los umbrales de valores imposibles y las medianas para imputar) se calcula **solo con train** (hasta 2022-10-27, 820 días), usando el mismo corte cronológico 70/15/15 que `temporal_split()` en `src/preprocessing_pipeline.py`. Validación, test y los datos nuevos de `predict.py` se transforman con esos valores, sin recalcular nada.

El frío de hoy no lo excluimos: es frio_hoy_d de la 2.2A y es la mejor feature. Los Mycom y Sala Maq no son leakage (son del día D), pero son redundantes: la suma de Mycom tiene 0,99 con el frío del mismo día, y su correlación con el target (0,83) es igual a la del frío de hoy (0,84), así que no aportan información nueva y meten multicolinealidad. Si quieren, en la Fase 3 probamos el modelo con y sin Suma Mycom y decidimos por el MAE de validación.

In [13]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import requests


from src.auxiliar_functions import RAW_DATA_DIR

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
from src.auxiliar_functions import PROCESSED_DATA_DIR

COL_TARGET = "Frio (Kw)_Consolidado EE"
df_horario = pd.read_csv(PROCESSED_DATA_DIR / "dataset_trabajo.csv", parse_dates=["DIA"], low_memory=False)

# Cierre diario (23:59) = total del día de cada totalizador
df_diario = (df_horario[df_horario["HORA"] == "23:59"]
             .drop(columns="HORA").sort_values("DIA").reset_index(drop=True).copy())

# Target D+1 por fecha calendario (si falta el día siguiente, queda NaN)
frio_por_dia = df_diario.set_index("DIA")[COL_TARGET]
df_diario["y"] = (df_diario["DIA"] + pd.Timedelta(days=1)).map(frio_por_dia)

# Mismo corte cronológico 70/15/15 que temporal_split() de src/preprocessing_pipeline.py
n = len(df_diario)
n_train, n_val = round(n * 0.70), round(n * 0.15)
FIN_TRAIN = df_diario["DIA"].iloc[n_train - 1]
FIN_VAL = df_diario["DIA"].iloc[n_train + n_val - 1]
es_train = df_diario["DIA"] <= FIN_TRAIN

print(f"Días: {n} ({df_diario['DIA'].min().date()} a {df_diario['DIA'].max().date()}) | columnas: {df_diario.shape[1]}")
print(f"Train hasta {FIN_TRAIN.date()} ({es_train.sum()} días) | Val hasta {FIN_VAL.date()} | Test hasta {df_diario['DIA'].max().date()}")

Días: 1172 (2020-07-01 a 2023-10-25) | columnas: 421
Train hasta 2022-10-27 (820 días) | Val hasta 2023-04-28 | Test hasta 2023-10-25


In [14]:
CANDIDATAS = [
    "Hl Producido Bodega_Consolidado Produccion", "Hl Cerveza Envasada_Consolidado Produccion",
    "Hl Cerveza Filtrada_Consolidado Produccion", "Hl de Mosto_Consolidado Produccion",
    "Hl Cerveza Cocina_Consolidado Produccion", "Hl Cerveza L4_Consolidado Produccion",
    "Agua Planta (Hl)_Consolidado Agua", "Planta de agua (Hl)_Consolidado Agua",
    "Produccion (Hl)_Consolidado Agua", "Agua Cond Evaporativos_Consolidado Agua",
    "Agua Envasado (Hl)_Consolidado Agua", "Agua Elaboracion (Hl)_Consolidado Agua",
    "Vapor Envasado (Kg)_Consolidado GasVapor", "Tot_Vapor_L3_L4_Consolidado GasVapor",
    "Elaboracion (Kw)_Consolidado EE", "Aire (Kw)_Consolidado EE", "Bodega (Kw)_Consolidado EE",
    "Envasado (Kw)_Consolidado EE",
    "KW Cond 5. 6 y 9_Totalizadores Energia", "KW Trafo 10_Totalizadores Energia",
    "KW Trafo 11_Totalizadores Energia", "KW Trafo 9_Totalizadores Energia",
    "Kw Compresores Aire_Totalizadores Energia",
    "Aire Producido (M3)_Consolidado Aire", "Aire Envasado (M3)_Consolidado Aire",
    "Aire Planta (M3)_Consolidado Aire",
    "Totalizador_Filtracion_Totalizadores CO2", "Totalizador_Tq_Pulmon_Totalizadores CO2",
]
# Proxies del target (Sección 1.3): contienen el Frío del mismo día -> no son candidatas
PROXIES = ["Sala Maq (Kw)_Consolidado EE", "Servicios (Kw)_Consolidado EE", "Planta (Kw)_Consolidado EE",
           "KW Gral Planta_Totalizadores Energia"] + [f"KW Mycom {i}_Totalizadores Energia" for i in range(1, 8)]
assert not set(CANDIDATAS) & set(PROXIES), "Hay un proxy del target entre las candidatas"

MAX_NULOS_TRAIN = 0.20   # más de 20 % de nulos en train: no hay base suficiente para imputar
UMBRAL_COLINEAL = 0.90   # |Spearman| a partir del cual dos predictoras son redundantes
MIN_RHO_TRAIN = 0.40     # relación mínima con el target, calculada SOLO en train


def seleccionar_representantes(df, columnas, col_y="y", max_nulos=MAX_NULOS_TRAIN,
                               umbral=UMBRAL_COLINEAL, min_rho=MIN_RHO_TRAIN):

    """Elige una variable por grupo colineal usando SOLO el DataFrame que recibe (train).

    1. Descarta columnas con más de `max_nulos` de nulos.
    2. Ordena las restantes por |Spearman| con el target, de mayor a menor.
    3. Conserva una variable solo si su |Spearman| con todas las ya conservadas es menor
       a `umbral`; si no, es redundante con una que se relaciona mejor con el target.
    """
    nulos = df[columnas].isna().mean()
    rho_y = df[columnas].corrwith(df[col_y], method="spearman")
    corr = df[columnas].corr(method="spearman").abs()

    decision, elegidas = {}, []
    for col in rho_y.abs().sort_values(ascending=False).index:
        if nulos[col] > max_nulos:
            decision[col] = f"descartada: {nulos[col]:.0%} de nulos en train"
            continue
        if abs(rho_y[col]) < min_rho:
            decision[col] = f"descartada: |Spearman| con el target en train < {min_rho}"
            continue
        redundante_con = [e for e in elegidas if corr.loc[col, e] >= umbral]
        if redundante_con:
            decision[col] = f"redundante con {redundante_con[0]} (|rho| = {corr.loc[col, redundante_con[0]]:.3f})"
        else:
            elegidas.append(col)
            decision[col] = "elegida"
    

    tabla = pd.DataFrame({"spearman_y_train": rho_y.round(3), "nulos_train": nulos.round(3),
                          "decision": pd.Series(decision)}).sort_values("spearman_y_train", key=abs, ascending=False)
    return elegidas, tabla


PREDICTORAS, tabla_seleccion = seleccionar_representantes(df_diario[es_train], CANDIDATAS)
print(f"Predictoras elegidas: {len(PREDICTORAS)} de {len(CANDIDATAS)} candidatas")
tabla_seleccion

Predictoras elegidas: 15 de 28 candidatas


,spearman_y_train,nulos_train,decision
KW Trafo 11_Totalizadores Energia,0.708,0.000,elegida
Agua Planta (Hl)_Consolidado Agua,0.688,0.000,elegida
Planta de agua (Hl)_Consolidado Agua,0.675,0.000,redundante con Agua Planta (Hl)_Consolidado Ag...
KW Cond 5. 6 y 9_Totalizadores Energia,0.673,0.000,elegida
Produccion (Hl)_Consolidado Agua,0.635,0.000,redundante con Agua Planta (Hl)_Consolidado Ag...
KW Trafo 9_Totalizadores Energia,0.590,0.000,elegida
KW Trafo 10_Totalizadores Energia,0.555,0.000,elegida
Aire Producido (M3)_Consolidado Aire,0.544,0.000,elegida
Kw Compresores Aire_Totalizadores Energia,0.542,0.000,redundante con Aire Producido (M3)_Consolidado...
Agua Envasado (Hl)_Consolidado Agua,0.530,0.000,elegida


**Selección de un representante por grupo colineal.** Se parte de las 28 candidatas de la Sección 1.3 y se excluyen los proxies del target (`Sala Maq`, `Servicios`, `Planta`, `KW Gral Planta` y los compresores `KW Mycom`). La función `seleccionar_representantes` aplica, **calculando todo solo con train**:
1. Descarta columnas con más de 20 % de nulos.
2. Descarta columnas con |Spearman| con el target menor a 0,4.
3. Ordena las restantes por |Spearman| con el target y conserva una variable solo si su |Spearman| con todas las ya elegidas es menor a 0,9.

El umbral de 0,4 se recalcula en train porque en la exploración (Sección 1.3) se había calculado con todos los datos, incluidos validación y test. Así, toda decisión que depende del target se toma sin información del futuro. Con este criterio quedan afuera 7 candidatas que superaban 0,4 con todos los datos pero no en train.

Resultado: **15 predictoras de 28**.
- **Grupos redundantes:**
  - Agua total: se conserva `Agua Planta (Hl)`; salen `Planta de agua (Hl)` (0,95) y `Produccion (Hl)` (0,91).
  - Aire comprimido: se conserva `Aire Producido (M3)`; salen `Kw Compresores Aire` (0,99) y `Aire (Kw)` (0,98).
  - Envasado: se conserva `Envasado (Kw)`; sale `Hl Cerveza Envasada` (0,92).
- **Descartadas por nulos:** `Agua Cond Evaporativos`, que no existe en el archivo 2020_2022 (43 % de días sin dato en train).
- **Descartadas por relación débil en train (< 0,4):** `Vapor Envasado (Kg)` (0,38), `Aire Planta (M3)` (0,38), `Totalizador_Filtracion` (0,35), `Hl Cerveza Cocina` (0,34), `Hl de Mosto` (0,33), `Tot_Vapor_L3_L4` (0,32) y `Hl Cerveza L4` (0,25). Los grupos de cocina y vapor quedan sin representante.

In [15]:
from sklearn.pipeline import Pipeline
from src.limpieza_predictoras import LimpiadorExtremos, ImputadorTemporal

limpieza_predictoras = Pipeline([
    ("extremos", LimpiadorExtremos(factor=5.0, cuantil=0.99)),
    ("imputacion", ImputadorTemporal(limite=3)),
])

# fit SOLO con train: los topes (5 x P99) y las medianas salen de train
limpieza_predictoras.fit(df_diario.loc[es_train, PREDICTORAS])

# transform sobre todo el período: val y test usan los valores aprendidos en train
X_limpio = limpieza_predictoras.transform(df_diario[PREDICTORAS])

extremos = limpieza_predictoras.named_steps["extremos"]
imputacion = limpieza_predictoras.named_steps["imputacion"]
resumen_21 = pd.DataFrame({
    "tope_5xP99_train": extremos.topes_.round(0),
    "nulos_originales": df_diario[PREDICTORAS].isna().sum(),
    "imposibles_a_NaN": extremos.reemplazos_,
    "imputados_ffill": imputacion.imputados_ffill_,
    "imputados_mediana_train": imputacion.imputados_mediana_,
})
print("Columnas con algún cambio:")
print(resumen_21[resumen_21.iloc[:, 1:].sum(axis=1) > 0].to_string())
print(f"\nNaN restantes: {int(X_limpio.isna().sum().sum())} | X_limpio: {X_limpio.shape}")

# Qué valores se consideraron imposibles y en qué partición caen
particion = np.where(es_train, "train", np.where(df_diario["DIA"] <= FIN_VAL, "val", "test"))
es_imposible = (df_diario[PREDICTORAS] < 0) | (df_diario[PREDICTORAS] > extremos.topes_)
print("\nImposibles por partición:", es_imposible.groupby(particion).sum().sum(axis=1).to_dict())
for col in PREDICTORAS:
    if es_imposible[col].any():
        print(f"  {col}: {df_diario.loc[es_imposible[col], ['DIA', col]].values.tolist()}")

Columnas con algún cambio:
                                         tope_5xP99_train  nulos_originales  imposibles_a_NaN  imputados_ffill  imputados_mediana_train
Elaboracion (Kw)_Consolidado EE                   54049.0                 0                 1                1                        0
Envasado (Kw)_Consolidado EE                     107225.0                 0                 1                1                        0
Bodega (Kw)_Consolidado EE                        47656.0                 0                 6                6                        0
Totalizador_Tq_Pulmon_Totalizadores CO2          100804.0                 1                 1                2                        0

NaN restantes: 0 | X_limpio: (1172, 15)

Imposibles por partición: {'test': 0, 'train': 7, 'val': 2}
  Elaboracion (Kw)_Consolidado EE: [[Timestamp('2021-11-02 00:00:00'), -672.0]]
  Envasado (Kw)_Consolidado EE: [[Timestamp('2021-11-29 00:00:00'), 4504000.0]]
  Bodega (Kw)_Consolidado EE: [

**Limpieza de valores imposibles e imputación** (`src/limpieza_predictoras.py`), como transformadores de sklearn para reutilizarlos en el Pipeline y en `predict.py`:

1. `LimpiadorExtremos` pasa a `NaN` los valores **negativos** (regla física: un totalizador diario no puede ser negativo, independiente de los datos) y los mayores a **5 × P99 de train** (umbral estadístico, aprendido en `fit` solo con train).
2. `ImputadorTemporal` completa los `NaN` con **forward fill limitado a 3 días** (copia el último cierre anterior, nunca uno posterior) y lo que quede con la **mediana de train**.

Resultado: **9 valores imposibles** (7 en train, 2 en validación, 0 en test) y 1 nulo original (`Totalizador_Tq_Pulmon`), todos completados con forward fill; quedan 0 nulos. Los casos detectados:
- `Bodega (Kw)`: 6 cierres negativos (entre −236 y −6.247 kW).
- `Envasado (Kw)`: 4.504.000 kW el 2021-11-29 (su tope en train es 107.225 kW).
- `Elaboracion (Kw)`: −672 kW el 2021-11-02.
- `Totalizador_Tq_Pulmon`: −805.849 el 2020-08-28. Este valor explica la diferencia entre su Pearson y su Spearman observada en la exploración de *Totalizadores CO2*.

La limpieza no escala: los valores quedan en sus unidades originales, de acuerdo con la consigna para `dataset_final.csv`. El escalado se ajusta con `X_train` en `build_scaler_pipeline()`.

In [16]:
TECATE_LAT = 32.5722
TECATE_LON = -116.6283

def obtener_clima_tecate(
    fecha_inicio: str,
    fecha_fin: str,
    ruta_cache: Path = RAW_DATA_DIR / "clima_tecate_openmeteo.csv",
) -> pd.DataFrame:
    """
    Descarga desde Open-Meteo (o lee de caché local) las temperaturas diarias
    y la radiación solar acumulada de Tecate para el rango [fecha_inicio, fecha_fin].
    """
    req_ini = pd.to_datetime(fecha_inicio)
    req_fin = pd.to_datetime(fecha_fin)
    columnas_esperadas = [
        "DIA",
        "temp_max_c",
        "temp_min_c",
        "temp_media_c",
        "temp_aparente_c",
        "radiacion_mj_m2", #basicamente mide la energia solar que hay en ese dia, es como decir, el sol que pega en la planta y que tanta energia absorve (por ejemplo se calientan las tuberias)
    ]

    # Si el CSV local ya tiene todas las columnas (incluida radiación) y cubre las fechas, se reutiliza
    if ruta_cache.exists():
        df_cache = pd.read_csv(ruta_cache, parse_dates=["DIA"])
        if (
            set(columnas_esperadas).issubset(df_cache.columns)
            and df_cache["DIA"].min() <= req_ini
            and df_cache["DIA"].max() >= req_fin
        ):
            mask = df_cache["DIA"].between(req_ini, req_fin)
            return df_cache.loc[mask, columnas_esperadas].sort_values("DIA").reset_index(drop=True)

    # Consulta a la API histórica de Open-Meteo
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": TECATE_LAT,
        "longitude": TECATE_LON,
        "start_date": req_ini.strftime("%Y-%m-%d"),
        "end_date": req_fin.strftime("%Y-%m-%d"),
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "temperature_2m_mean",
            "apparent_temperature_mean",
            "shortwave_radiation_sum",
        ],
        "timezone": "America/Tijuana",
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    daily = resp.json()["daily"]

    df_clima = pd.DataFrame({
        "DIA": pd.to_datetime(daily["time"]),
        "temp_max_c": daily["temperature_2m_max"],
        "temp_min_c": daily["temperature_2m_min"],
        "temp_media_c": daily["temperature_2m_mean"],
        "temp_aparente_c": daily["apparent_temperature_mean"],
        "radiacion_mj_m2": daily["shortwave_radiation_sum"],
    }).sort_values("DIA").reset_index(drop=True)

    # Guardamos en caché con salto de línea '\n' para mantener el mismo formato en Windows y Linux
    df_clima.to_csv(ruta_cache, index=False, lineterminator="\n")
    return df_clima

def agregar_features_clima(df_clima: pd.DataFrame) -> pd.DataFrame:
    df_c = df_clima.sort_values("DIA").copy()

    # Amplitud térmica dentro del mismo día D
    df_c["amplitud_termica_c"] = df_c["temp_max_c"] - df_c["temp_min_c"]

    # Variables de lag (día anterior D-1) y promedios móviles (últimos 3 y 7 días hasta D)
    df_c["temp_media_lag1"] = df_c["temp_media_c"].shift(1)
    df_c["temp_media_roll3"] = df_c["temp_media_c"].rolling(window=3, min_periods=1).mean()
    df_c["temp_media_roll7"] = df_c["temp_media_c"].rolling(window=7, min_periods=1).mean()
    df_c["temp_max_roll3"] = df_c["temp_max_c"].rolling(window=3, min_periods=1).mean()
    df_c["radiacion_roll3"] = df_c["radiacion_mj_m2"].rolling(window=3, min_periods=1).mean()
    df_c["radiacion_roll7"] = df_c["radiacion_mj_m2"].rolling(window=7, min_periods=1).mean()

    return df_c

In [17]:
df_clima = agregar_features_clima(obtener_clima_tecate(df_diario["DIA"].min(), df_diario["DIA"].max()))
df_diario = df_diario.merge(df_clima[["DIA", "temp_media_c", "radiacion_roll7"]], on="DIA", how="left")
print("Días sin clima:", df_diario[["temp_media_c", "radiacion_roll7"]].isna().any(axis=1).sum())

Días sin clima: 0


In [18]:
from src.features_planta import FeaturesPlanta, COLUMNAS_BASE

features_planta = FeaturesPlanta(columnas_clima=("temp_media_c", "radiacion_roll7"))
features_planta.fit(df_diario[es_train])           # topes, medianas y Meta por mes: solo train
X_planta = features_planta.transform(df_diario)    # train, val y test con lo aprendido en train

particion = np.where(es_train, "train", np.where(df_diario["DIA"] <= FIN_VAL, "val", "test"))
tabla_22b = pd.DataFrame({
    "spearman_y_train": X_planta[es_train].corrwith(df_diario.loc[es_train, "y"], method="spearman"),
    "nulos_train_%": X_planta[es_train].isna().mean() * 100,
    "nulos_val_%": X_planta[particion == "val"].isna().mean() * 100,
    "nulos_test_%": X_planta[particion == "test"].isna().mean() * 100,
    "mediana": X_planta.median(),
    "max": X_planta.max(),
}).round(3).sort_values("spearman_y_train", key=abs, ascending=False)

print("Meta EE Frio típica por mes (train):", features_planta.meta_por_mes_.round(2).to_dict())
print("Valores imposibles a NaN:", features_planta.limpieza_.named_steps["extremos"].reemplazos_.to_dict())
tabla_22b

Meta EE Frio típica por mes (train): {1: 3.01, 2: 2.51, 3: 2.43, 4: 2.43, 5: 2.47, 6: 2.47, 7: 3.23, 8: 3.35, 9: 2.93, 10: 2.58, 11: 2.55, 12: 3.01}
Valores imposibles a NaN: {'Frio (Kw)_Consolidado EE': 0, 'Servicios (Kw)_Consolidado EE': 0, 'Bodega (Kw)_Consolidado EE': 6, 'Elaboracion (Kw)_Consolidado EE': 1, 'Agua Planta (Hl)_Consolidado Agua': 0, 'Hl Producido Bodega_Consolidado Produccion': 0, 'EE Frio / Hl_Consolidado KPI': 3}


,spearman_y_train,nulos_train_%,nulos_val_%,nulos_test_%,mediana,max
bodega_x_radiacion_roll7,0.767,0.000,0.0,0.0,176834.440,452413.331
ratio_frio_servicios,0.714,0.000,0.0,0.0,0.608,1.799
bodega_x_temp_media_c,0.690,0.000,0.0,0.0,145240.342,440858.580
eficiencia_frio_vs_meta,0.301,0.000,0.0,0.0,0.998,11.544
ratio_aguacond_aguaplanta,-0.121,43.415,0.0,0.0,0.053,0.249
ratio_bodega_elaboracion,0.036,0.000,0.0,0.0,0.851,25.923
meta_frio_imputada,0.034,0.000,0.0,0.0,0.000,1.000


# Sección 2.2 (parte B) — Features de planta: ratios, eficiencia contra la Meta e interacciones

Implementadas en `src/features_planta.py` como el transformador `FeaturesPlanta`. En `fit()` (solo con train) aprende los topes de valores imposibles y las medianas de las columnas base, y la Meta típica de cada mes; en `transform()` calcula las features. Todas usan datos del día D o anteriores.

- **Ratios entre áreas** (sobre columnas ya limpias; si el denominador es ≤ 0 el resultado es `NaN`): `ratio_frio_servicios`, `ratio_bodega_elaboracion` y `ratio_aguacond_aguaplanta`.
- **Eficiencia contra la Meta:** `EE Frio / Hl` ÷ `Meta EE Frio` (> 1 = más frío por hectolitro que lo planificado). `EE Frio / Hl` tenía 25 nulos y picos de hasta 878 por divisiones por una producción casi nula: con el tope de train (5 × P99 = 38,45) se pasaron 3 valores a `NaN` y se imputaron. Las metas son mensuales y solo existen desde julio de 2021; para los meses anteriores se usa la Meta típica del mismo mes aprendida en train (de 2,43 kWh/Hl en marzo-abril a 3,35 en agosto), y la columna `meta_frio_imputada` marca esos días.
- **Interacciones producción × clima:** `Hl Producido Bodega` multiplicado por la temperatura media y por la radiación solar promedio de 7 días (clima de Tecate de la Sección 2.2A).

**Resultados (Spearman con el target en train):**

| Feature | Spearman | Decisión |
|---|---|---|
| `bodega_x_radiacion_roll7` | 0,767 | Candidata: supera a sus dos factores por separado (`Hl Producido Bodega`: 0,52; `radiacion_roll7`: ~0,74). La carga de frío depende de la combinación de mucha cerveza en bodega y calor |
| `ratio_frio_servicios` | 0,714 | Candidata; contiene el Frío del día D, a evaluar su redundancia con `frio_hoy_d` en la Sección 2.3 |
| `bodega_x_temp_media_c` | 0,690 | Candidata; probable redundancia con la interacción de radiación |
| `eficiencia_frio_vs_meta` | 0,301 | Se conserva por ser requerida por la consigna; relación débil con el target |
| `ratio_aguacond_aguaplanta` | −0,121 | Descartada: 43 % de nulos en train (`Agua Cond Evaporativos` no existe en el archivo 2020_2022) |
| `ratio_bodega_elaboracion` | 0,036 | Descartada: sin relación con el target |
| `meta_frio_imputada` | — | Indicador de trazabilidad, no se usa como predictora |